<a href="https://colab.research.google.com/github/yilinw762/ds2002-fa26/blob/main/notebooks/04-json-apis/2026_10_09_%E2%80%94_Public_API_to_Clean_Table_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [2]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    if tries < 1:
        raise ValueError("tries must be at least 1")

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start,
        "end_date": end,
        "daily": "temperature_2m_max,precipitation_sum",
        "timezone": "America/New_York",
    }
    # TODO: loop up to `tries` times
    for attempt in range(tries):
        try:
    # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
            r = requests.get(BASE, params=params, timeout=15)
    # TODO:   raise on a status that is not in RETRY_STATUSES
        except (requests.Timeout, requests.ConnectionError):
            if attempt == tries - 1:
                raise
    # TODO:   otherwise sleep (increasing each time) and try again
        else:
            if r.status_code == 200:
                return pd.DataFrame(r.json()["daily"])

            if r.status_code != 429 and not 500 <= r.status_code < 600:
                r.raise_for_status()
                raise RuntimeError(f"Unexpected status: {r.status_code}")

            if attempt == tries - 1:
                r.raise_for_status()

        time.sleep(2 ** attempt)


### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [3]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
from datetime import datetime, timezone

target_dates = ["2026-09-05", "2026-10-03"]
used_fallback = False
pulled_at = datetime.now(timezone.utc)

try:
    wx = get_weather(target_dates[0], target_dates[1])
except requests.HTTPError as exc:
    # Invalid requests should fail loudly.
    if exc.response.status_code != 429 and not 500 <= exc.response.status_code < 600:
        raise

    print(f"API failed after retries: {exc}")
    used_fallback = True
except (requests.Timeout, requests.ConnectionError) as exc:
    print(f"Network failed after retries: {exc}")
    used_fallback = True
else:
    wx = wx.loc[wx["time"].isin(target_dates)].copy()

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [4]:
FALLBACK = pd.DataFrame({
    "time": ["2026-09-05", "2026-10-03"],
    "temperature_2m_max": [30.1, 18.4],
    "precipitation_sum": [0.0, 22.5],
})

if used_fallback:
    wx = FALLBACK.copy()

wx["source"] = "fallback sample" if used_fallback else "open-meteo"
wx
# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [5]:
# TODO
wx = wx.rename(columns={
    "time": "date",
    "temperature_2m_max": "temp_max",
    "precipitation_sum": "precip_mm",
})

wx["date"] = pd.to_datetime(wx["date"], errors="raise")
wx["temp_max"] = pd.to_numeric(wx["temp_max"], errors="raise")
wx["precip_mm"] = pd.to_numeric(wx["precip_mm"], errors="raise")

wx = (
    wx[["date", "temp_max", "precip_mm", "source"]]
    .sort_values("date")
    .reset_index(drop=True)
)

wx

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [6]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [7]:
# TODO
rain = wx.set_index("date")["precip_mm"]
first, second = pd.to_datetime(target_dates)

difference = abs(rain.loc[first] - rain.loc[second])
label = "fallback sample data" if used_fallback else "Open-Meteo data"

if difference == 0:
    print(
        f"Using {label}, September 5 and October 3, 2026 "
        f"had the same rainfall: {rain.loc[first]:.1f} mm."
    )
else:
    wetter = first if rain.loc[first] > rain.loc[second] else second
    drier = second if wetter == first else first

    print(
        f"Using {label}, {wetter:%B %d, %Y} was wetter than "
        f"{drier:%B %d, %Y} by {difference:.1f} mm of rainfall."
    )

Using Open-Meteo data, October 03, 2026 was wetter than September 05, 2026 by 12.2 mm of rainfall.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [8]:
# TODO
print(f"Endpoint: {BASE}")
print("Coordinates: latitude 38.03, longitude -78.51")
print("Requested date range: 2026-09-05 through 2026-10-03")
print(f"Retrieval attempted at: {pulled_at.isoformat()}")
print(f"Data source: {'fallback sample' if used_fallback else 'real Open-Meteo data'}")

Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: latitude 38.03, longitude -78.51
Requested date range: 2026-09-05 through 2026-10-03
Retrieval attempted at: 2026-10-09T01:40:04.370549+00:00
Data source: real Open-Meteo data


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?


An empty weather table could silently distort the analysis. A left join would preserve the 200 sales rows but leave rainfall missing, so the chart might omit those rows and appear blank. If missing rainfall were replaced with zero, it would falsely show every day as dry. An inner join would remove all sales rows. Nobody might catch the problem because the pipeline could finish without an error, and missing data would be mistaken for valid results unless row counts, missing values, and data sources were checked.

_your answer here_

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [9]:
# stretch
orlando = get_weather(
    "2026-09-05",
    "2026-10-03",
    lat=28.54,
    lon=-81.38,
)

orlando = orlando.loc[orlando["time"].isin(target_dates)].copy()

orlando = orlando.rename(columns={
    "time": "date",
    "temperature_2m_max": "temp_max",
    "precipitation_sum": "precip_mm",
})

orlando["date"] = pd.to_datetime(orlando["date"], errors="raise")
orlando["temp_max"] = pd.to_numeric(orlando["temp_max"], errors="raise")
orlando["precip_mm"] = pd.to_numeric(orlando["precip_mm"], errors="raise")
orlando["source"] = "open-meteo"

both_cities = pd.concat(
    [
        wx.assign(city="Charlottesville"),
        orlando.assign(city="Orlando"),
    ],
    ignore_index=True,
)

both_cities = (
    both_cities[["city", "date", "temp_max", "precip_mm", "source"]]
    .sort_values(["city", "date"])
    .reset_index(drop=True)
)

assert len(both_cities) == 4
assert not both_cities.duplicated(["city", "date"]).any()
assert both_cities["precip_mm"].notna().all()
assert (both_cities["precip_mm"] >= 0).all()

both_cities

,city,date,temp_max,precip_mm,source
0,Charlottesville,2026-09-05,29.4,2.5,open-meteo
1,Charlottesville,2026-10-03,22.0,14.7,open-meteo
2,Orlando,2026-09-05,32.1,3.9,open-meteo
3,Orlando,2026-10-03,30.4,14.5,open-meteo
